# Ridge regression: the maths and code from scratch

In [1]:
import numpy as np
from sklearn.datasets import load_diabetes, make_regression
from sklearn.linear_model import Ridge
from sklearn.metrics import r2_score
from sklearn.model_selection import train_test_split

## One input: closed form for m and b

In [2]:
class MyRidge:
    def __init__(self, alpha=0.1):
        self.alpha = alpha

    def fit(self, x, y):
        num = np.sum((x - x.mean()) * (y - y.mean()))
        den = np.sum((x - x.mean()) ** 2) + self.alpha
        self.m = num / den
        self.b = y.mean() - self.m * x.mean()
        return self

X, y = make_regression(n_samples=100, n_features=1, n_informative=1, noise=20, random_state=13)
x = X.ravel()
for alpha in [0, 10, 100]:
    mine = MyRidge(alpha).fit(x, y)
    sk = Ridge(alpha=alpha).fit(X, y)
    print(f"alpha {alpha:3d}: mine m={mine.m:.3f} b={mine.b:.3f} | sklearn m={sk.coef_[0]:.3f} b={sk.intercept_:.3f}")

alpha   0: mine m=27.828 b=-2.295 | sklearn m=27.828 b=-2.295
alpha  10: mine m=24.955 b=-2.127 | sklearn m=24.955 b=-2.127
alpha 100: mine m=12.934 b=-1.425 | sklearn m=12.934 b=-1.425


In [3]:
# The two sums of the slope formula on the 100 observations (Section 2.4)
xs, ys = X.ravel(), y
print("top    sum (x - xbar)(y - ybar) =", round(np.sum((xs - xs.mean()) * (ys - ys.mean())), 1))
print("bottom sum (x - xbar)^2         =", round(np.sum((xs - xs.mean()) ** 2), 2))

# Two points (1, 2) and (3, 5), lambda = 1, by the matrix formula (Section 3.5)
X2 = np.array([[1.0, 1.0], [1.0, 3.0]]); y2 = np.array([2.0, 5.0])
I0 = np.diag([0.0, 1.0])
w2 = np.linalg.inv(X2.T @ X2 + 1.0 * I0) @ X2.T @ y2
print("two points: intercept, slope =", w2.round(3), "| one-feature formula:", MyRidge(alpha=1).fit(X2[:, 1], y2).m)

top    sum (x - xbar)(y - ybar) = 2416.7
bottom sum (x - xbar)^2         = 86.85
two points: intercept, slope = [1.5 1. ] | one-feature formula: 1.0


## Many inputs: (X^T X + lambda I)^-1 X^T y

In [4]:
class MyRidgeND:
    def __init__(self, alpha=0.1):
        self.alpha = alpha

    def fit(self, X, y):
        X = np.insert(X, 0, 1, axis=1)          # column of 1s for the intercept
        I = np.identity(X.shape[1])
        I[0, 0] = 0                             # do not penalise the intercept
        beta = np.linalg.inv(X.T @ X + self.alpha * I) @ X.T @ y
        self.intercept_, self.coef_ = beta[0], beta[1:]
        return self

    def predict(self, X):
        return X @ self.coef_ + self.intercept_

X, y = load_diabetes(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=4)
mine = MyRidgeND(alpha=0.1).fit(X_train, y_train)
sk = Ridge(alpha=0.1, solver="cholesky").fit(X_train, y_train)
print("intercept", round(mine.intercept_, 2), round(sk.intercept_, 2))
print("test R2  ", round(r2_score(y_test, mine.predict(X_test)), 4), round(sk.score(X_test, y_test), 4))
print("max coef difference", np.abs(mine.coef_ - sk.coef_).max())

intercept 150.89 150.89
test R2   0.4693 0.4693
max coef difference 2.9274360713316128e-12
